# 10g Licence-cleared decoding examples (NSD)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a and 10b.

Development analysis, not a prespecified test. NSD's terms forbid distributing its images, so no picture comes from NSD's stimulus files. Each test image is
mapped to its COCO id, its Flickr licence read from COCO's annotation files, and pictures are downloaded from COCO's
servers. Seen images are drawn at random (seed 370) among gallery images with a reusable licence (no known
copyright, US government work, CC BY); guesses with other licences become grey tiles. Decoding uses 10f's code for
subj01, draw 13, k = 500. Writes `10g_examples.json`: each system's top-5 guesses with licence, COCO URL and
caption, for the paper's figure.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10g-licensedexamples', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "draws": [13], "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "wall_hours": 11.0, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "ks": [500], "example_subject": "subj01", "example_k": 500, "example_draw": 13, "example_images": 6, "example_seed": 370, "stim_info": "https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_stim_info_merged.csv", "coco_annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "reusable_licences": ["No known copyright restrictions", "United States Government Work", "Attribution License"]}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    # the paper's OLS (intercept, normal equations), per output, in float64
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {}
for name in ("alexnet", "dinov2"):
    for s in SUBJECTS:
        BETA[(name, s)] = ols_coef(feats(name, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(name, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
METHODS = ['avg', 'lowrank_map', 'own_ridge', 'stack', 'avg_dinov2', 'mixed']
def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        keep = np.setdiff1d(np.arange(k), held)
        pred = fit_predict(keep, held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

def ranks(pred, meas):
    # per gallery: rank of the true image (0 = identified) and the 5 best candidates (positions in test_ids), raw Pearson
    out_rank, out_top = np.full(len(meas), -1), np.full((len(meas), 5), -1)
    for g in GALS:
        p, m = pred[g], meas[g]
        p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
        m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
        r = m @ p.T / p.shape[1]
        out_rank[g] = (r > np.diag(r)[:, None]).sum(1)
        out_top[g] = np.asarray(g)[np.argsort(-r, 1)[:, :5]]
    return out_rank, out_top

in_gal = np.zeros(len(test_ids), bool)
for g in GALS:
    in_gal[g] = True
VERT = {m: np.zeros((len(SUBJECTS), V), np.float32) for m in METHODS}
HIT = {m: np.zeros((len(SUBJECTS), len(test_ids)), np.float32) for m in METHODS}
examples = {}
for p, s in enumerate(SUBJECTS, start=1):
    if s != FIXED["example_subject"]:
        continue
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    meas = np.asarray(Y[s][test_rows[s]], np.float64)
    mc = meas - meas.mean(0)
    y_pool = np.asarray(Y[s][pool_rows[s]], np.float32)
    X = {f: (feats(f, ids[s][pool_rows[s]]), feats(f, test_ids)) for f in ("alexnet", "dinov2")}
    def donor_mean(f, x):
        acc = np.zeros((len(x), V), np.float32)
        for d in donors:
            acc += ols_pred(BETA[(f, d)], x).astype(np.float32)
        return acc / len(donors)
    avg = {f: (donor_mean(f, X[f][0]), donor_mean(f, X[f][1]).astype(np.float64)) for f in ("alexnet", "dinov2")}
    basis = {f: lm.prediction_basis(avg[f][0], FIXED["K"]) for f in ("alexnet", "dinov2")}
    count = 0
    def add(method, pred, keep_top=False):
        pc = pred - pred.mean(0)
        VERT[method][p - 1] += ((pc * mc).sum(0) / np.sqrt((pc ** 2).sum(0) * (mc ** 2).sum(0) + 1e-30)).astype(np.float32)
        rk, top = ranks(pred, meas)
        HIT[method][p - 1] += (rk == 0).astype(np.float32)
        if keep_top:
            examples.setdefault(method, top)
    for d in FIXED["draws"]:
        order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
        for k in FIXED["ks"]:
            idx = order[:k]
            y_k = y_pool[idx].astype(np.float64)
            keep = s == FIXED["example_subject"] and k == FIXED["example_k"] and d == FIXED["example_draw"]
            parts = {}
            for f in ("alexnet", "dinov2"):
                a_k, a_t = avg[f][0][idx].astype(np.float64), avg[f][1]
                mean, B = basis[f]
                m_test = lm.apply_lowrank_map(lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d), a_t, mean, B)
                m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), k, d)
                parts[("map", f)] = (m_oof, m_test)
            xk = X["alexnet"][0][idx]
            own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(xk, y_k)
            parts["own"] = (oof(lambda tr, he: own.refit(xk[tr], y_k[tr]).predict(xk[he]), k, d), own.predict(X["alexnet"][1]))
            stacks = {}
            for name, chosen in (("stack", [("map", "alexnet"), "own"]), ("mixed", [("map", "dinov2"), "own"])):
                w = nc.stack_fit([parts[c][0] for c in chosen], y_k)
                stacks[name] = nc.stack_apply(w, [parts[c][0] for c in chosen], [parts[c][1] for c in chosen], y_k.mean(0))
            add("avg", nc.with_labelled_mean(avg["alexnet"][1], avg["alexnet"][0][idx].astype(np.float64), y_k), keep)
            add("avg_dinov2", nc.with_labelled_mean(avg["dinov2"][1], avg["dinov2"][0][idx].astype(np.float64), y_k), keep)
            add("lowrank_map", parts[("map", "alexnet")][1], keep)
            add("own_ridge", parts["own"][1], keep)
            add("stack", stacks["stack"], keep)
            add("mixed", stacks["mixed"], keep)
            count += 1
    for m in METHODS:
        VERT[m][p - 1] /= count
        HIT[m][p - 1] /= count
    print(s, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")
    if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
        print("STOPPED: wall clock after", s)
        break

In [ ]:
import io
import urllib.request
import zipfile
from PIL import Image, ImageOps

def fetch(url, tries=4):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                return r.read()
        except Exception as e:
            print("retry", url, e)
            time.sleep(10 * (i + 1))
    raise RuntimeError(f"could not fetch {url}")

stim = pd.read_csv(io.BytesIO(fetch(FIXED["stim_info"])))
coco_of = {int(r.nsdId): (int(r.cocoId), str(r.cocoSplit)) for r in stim.itertuples()}  # 0-based, as data.nsd.image_ids
shared_flag = dict(zip(stim.nsdId.astype(int), stim.shared1000.astype(bool)))
if not all(shared_flag[int(i)] for i in test_ids):  # the 872 images all 8 people saw are among NSD's shared 1,000
    raise ValueError("NSD id mapping is off: test images are not all in shared1000")
print("id check passed: all", len(test_ids), "test images are NSD shared1000 images")
z = zipfile.ZipFile(io.BytesIO(fetch(FIXED["coco_annotations"])))
info, captions, licence_name = {}, {}, {}
for split in ("train2017", "val2017"):
    ann = json.loads(z.read(f"annotations/captions_{split}.json"))
    licence_name.update({l["id"]: l["name"] for l in ann["licenses"]})
    for im in ann["images"]:
        info[im["id"]] = im
    for c in ann["annotations"]:
        captions.setdefault(c["image_id"], c["caption"].strip())
print(len(info), "COCO images;", licence_name)

def meta(pos):
    coco_id, split = coco_of[int(test_ids[pos])]
    im = info[coco_id]
    return {"pos": int(pos), "nsd_id": int(test_ids[pos]), "coco_id": coco_id, "split": split,
            "licence": licence_name[im["license"]], "coco_url": im["coco_url"], "flickr_url": im.get("flickr_url", ""),
            "caption": captions.get(coco_id, "")}

gallery_pos = np.where(in_gal)[0]
metas = {int(i): meta(i) for i in range(len(test_ids))}
lic_counts = pd.Series([metas[i]["licence"] for i in gallery_pos]).value_counts().to_dict()
print("gallery licences:", lic_counts)
ok = [i for i in gallery_pos if metas[i]["licence"] in FIXED["reusable_licences"]]
if len(ok) < FIXED["example_images"]:
    raise RuntimeError(f"only {len(ok)} reusable gallery images")
pick = np.sort(np.random.default_rng(FIXED["example_seed"]).choice(ok, FIXED["example_images"], replace=False))

def picture(pos):
    m = metas[pos]
    img = Image.open(io.BytesIO(fetch(m["coco_url"]))).convert("RGB")
    return ImageOps.fit(img, (256, 256))  # centre crop to a square, as NSD shows a square crop

show = [("avg", "others' average"), ("stack", "map + own model"), ("mixed", "mixed (DINOv2 map)")]
fig, axes = plt.subplots(len(pick), 1 + 5 * len(show), figsize=(16, 1.3 * len(pick)), gridspec_kw={"wspace": 0.05, "hspace": 0.1})
credits, shown = {}, set()
for r_, i in enumerate(pick):
    cells = [(i, "seen image")] + [(examples[m][i][j], lab if j == 0 else "") for m, lab in show for j in range(5)]
    for c_, (pos, title) in enumerate(cells):
        ax = axes[r_, c_]
        ax.set_xticks([]); ax.set_yticks([])
        m = metas[int(pos)]
        if m["licence"] in FIXED["reusable_licences"]:
            ax.imshow(picture(int(pos)))
            credits[m["coco_id"]] = {k: m[k] for k in ("licence", "flickr_url", "coco_url")}
        else:
            ax.set_facecolor("#E6E6E6")
            ax.text(0.5, 0.5, "\n".join(m["caption"][i_:i_ + 22] for i_ in range(0, min(len(m["caption"]), 66), 22)),
                    ha="center", va="center", fontsize=4.6, color="#444444", transform=ax.transAxes)
        hit = c_ > 0 and int(pos) == int(i)
        for sp in ax.spines.values():
            sp.set_color("#C8102E" if hit else ("black" if c_ == 0 else "#DDDDDD")); sp.set_linewidth(2.5 if hit or c_ == 0 else 0.5)
        if r_ == 0 and title:
            ax.set_title(title, fontsize=8, loc="left")
fig.savefig(C.METRICS_DIR / "10g_decoding_examples.png", dpi=200, bbox_inches="tight")
fig.savefig(C.METRICS_DIR / "10g_decoding_examples.pdf", bbox_inches="tight")
summary = {"post_hoc": "development; one NSD person of P-7", "subject": FIXED["example_subject"], "k": 500, "draw": 13,
           "gallery_licences": lic_counts, "reusable_gallery_images": len(ok), "picked": [metas[int(i)] for i in pick],
           "hits_top1": {m: [bool(examples[m][i][0] == i) for i in pick] for m, _ in show},
           "rank_in_top5": {m: [int(list(examples[m][i]).index(i)) if i in examples[m][i] else -1 for i in pick] for m, _ in show},
           "credits": {str(k): v for k, v in credits.items()}}
(C.METRICS_DIR / "10g_summary.json").write_text(json.dumps(summary, indent=2))
# each system's top-5 guesses with licence, COCO URL and caption, so the paper figure can be laid
# out locally from COCO's servers; still no NSD stimulus file and no NSD response
guesses = {str(int(i)): {"seen": metas[int(i)], **{m: [metas[int(g)] for g in examples[m][i][:5]] for m, _ in show}} for i in pick}
(C.METRICS_DIR / "10g_examples.json").write_text(json.dumps(guesses, indent=1))
print("10G_SUMMARY_JSON " + json.dumps(summary))